# ST455 Group Project: Adaptive Q-learning Agent for Simplified Multi-player Texas Hold'em

This notebook implements and evaluates a tabular Q-learning agent in a multi-player limit Texas Hold'em environment. The environment of this project remains the core poker structure of private cards, community cards, fixed-limit betting rounds, folding, and showdown, while simplifying the betting size and side-pot logic.

This project has three main objectives:

1. Build a playable fixed-limit multi-player poker environment.
2. Train and evaluate a Q-learning player against rule-based opponents.
3. Test whether opponent-style tracking and adaptive Q-value adjustment can improve performance under time-varying opponent pools.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List,Tuple,Dict,Any,Optional
import random
from itertools import combinations
from collections import deque, Counter, defaultdict

## 1. Simplified Poker Environment

This section defines the poker environment that will be used for training and evaluation. The environment is simplified to make the reinforcement learning problem remains computationally manageable while still preserving the main decision structures of Texas Hold'em.


### 1.1 Card utilities
Cards are represented as `(rank, suit)`, where ranks range from 2 to 14 and suits are encoded as clubs, diamonds, hearts, and spades.


In [ ]:
# A card is represented as (rank, suit)
# rank: 2-14, where 11=J, 12=Q, 13=K, 14=A
# suit: C/D/H/S, representing Clubs, Diamonds, Hearts, Spades

Card = Tuple[int, str]

RANKS = list(range(2,15))
SUITS = ["C", "D", "H", "S"]

RANK_TO_STR = {
    11: "J",
    12: "Q",
    13: "K",
    14: "A"
}

# create a standard 52-card deck.
def create_deck() -> List[Card]:
    return [(rank, suit) for rank in RANKS for suit in SUITS]

# convert card into a readable string.
def card_to_str(card: Card) -> str:
    rank, suit =card
    rank_str = RANK_TO_STR.get(rank, str(rank))
    return f"{rank_str}{suit}"

# Convert a list of cards into readable strings.
def cards_to_str(cards: List[Card]) -> List[str]:
    return [card_to_str(card) for card in cards]

### 1.2 Player state
Each player stores private cards, stack size, folding status, all-in status, and current betting contribution.


In [ ]:
@dataclass
class PlayerState:
    player_id: int
    stack: int
    hole_cards: List[Card] = field(default_factory=list)
    folded: bool = False
    all_in: bool = False
    current_bet: int = 0
    total_contribution: int = 0

### 1.3 Action space
The agents can choose one from three discrete actions: fold, check/call, and bet/raise.


In [ ]:
ACTIONS = { 0: "fold",
    1: "check_call",
    2: "bet_raise"}

### 1.4 Hand evaluation
These functions evaluate five-card hands and select the best five-card combination from all available cards at showdown.


In [ ]:
# detect whether the current cards contain a four-card straight draw.
def _has_straight_draw(cards):
    ranks = set(card[0] for card in cards)

    if 14 in ranks:
        ranks.add(1)

    for start in range(1, 11):
        straight_window = set(range(start, start + 5))
        matched_ranks = ranks.intersection(straight_window)

        if len(matched_ranks) == 4:
            return True

    return False

# Estimate drawing potential after the flop.
def estimate_draw_strength(hole_cards, community_cards):
    cards = list(hole_cards) + list(community_cards)
    if len(community_cards)==0 or len(cards)<5:
        return 0

    best_score, _ = evaluate_best_hand(cards)
    hand_category = best_score[0]

    suits = [card[1] for card in cards]
    suit_counts = Counter(suits)

    flush_draw = max(suit_counts.values())==4
    straight_draw = _has_straight_draw(cards)

    if hand_category in {5,8}:
        flush_draw = False

    if hand_category in {4,8}:
        straight_draw = False

    if flush_draw and straight_draw:
        return 3
    if flush_draw:
        return 2
    if straight_draw:
        return 1

    return 0


# estimate hand strength using simple poker logic.
def estimate_simple_hand_strength(hole_cards, community_cards):
    cards = list(hole_cards) + list(community_cards)
    if len(community_cards) == 0:
        ranks = sorted([card[0] for card in hole_cards], reverse=True)
        suits = [card[1] for card in hole_cards]

        r1, r2 = ranks

        is_pair = r1 == r2
        is_suited = suits[0] == suits[1]
        gap = abs(r1 - r2)

        # pocket pairs
        if is_pair and r1 >= 10:
            return 4

        if is_pair:
            return 3

        # strong high-card hands
        if r1 == 14 and r2 >= 10:
            return 4

        if r1 >= 13 and r2 >= 10:
            return 3

        # suited broadway or near-broadway hands
        if is_suited and r1 >= 12 and r2 >= 9:
            return 3

        # suited connectors and high connectors
        if is_suited and gap <= 1 and r1 >= 10:
            return 2

        if gap <= 1 and r1 >= 11:
            return 2

        # medium high-card hands
        if r1 >= 12:
            return 2

        if r1 >= 10:
            return 1

        # low suited connectors have some playable potential
        if is_suited and gap <= 1 and r1 >= 8:
            return 1

        return 0

    if len(cards) >= 5:
        best_score, _ = evaluate_best_hand(cards)
        hand_category = best_score[0]

        draw_strength = estimate_draw_strength(
            hole_cards=hole_cards,
            community_cards=community_cards
        )

        if hand_category >= 6:
            return 4       # full house or better

        elif hand_category >= 4:
            return 3       # straight or flush

        elif hand_category >= 2:
            return 2       # two pair or three of a kind

        elif hand_category == 1:
            # one pair with a strong draw is treated as medium strength.
            if draw_strength >= 2:
                return 2

            return 1

        else:
            # High-card hand with drawing potential should not be treated as pure trash.
            if draw_strength == 3:
                return 2

            if draw_strength in {1, 2}:
                return 1

            return 0

    # fallback logic, mainly for safety.
    ranks = [card[0] for card in cards]
    rank_counts = Counter(ranks)
    max_count = max(rank_counts.values())

    if max_count >= 3:
        return 3

    if max_count == 2:
        return 1

    high_card = max(ranks)

    if high_card >= 13:
        return 1

    return 0

### 1.5 Environment configuration
The configuration controls player number, starting stack, blind size, fixed bet size, and maximum raises per betting round.


In [ ]:
@dataclass
class PokerConfig:
    n_players: int =4
    starting_stack: int =100

    small_blind: int =1
    big_blind: int =2
    fixed_bet: int =2

    max_raises_per_round: int =2

    def __post_init__(self):
        if self.n_players<2:
            raise ValueError("n_players must be at least 2.")
        if self.starting_stack<=0:
            raise ValueError("starting_stack must be positive.")
        if self.small_blind<=0 or self.big_blind<=0:
            raise ValueError("Blinds must be positive.")
        if self.big_blind<self.small_blind:
            raise ValueError("big_blind should be at least small_blind.")

### 1.6 Environment implementation
The environment handles dealing, blind posting, fixed-limit betting, round transitions, terminal states, showdown evaluation, and terminal rewards.


In [ ]:
class FixedLimitTexasHoldemEnv:

    #initialize the poker environment
    def __init__(self, config: Optional[PokerConfig] = None):
        self.config = config if config is not None else PokerConfig()
        self.rng = random.Random()

        self.dealer_button = -1
        self.players: List[PlayerState] = []

        self.deck: List[Card] = []
        self.community_cards: List[Card] = []

        self.pot = 0
        self.current_bet = 0
        self.current_player = None
        self.round_name = "preflop"

        self.raise_count = 0
        self.acted_this_round = set()

        self.done = False
        self.terminal_reason = None
        self.winner_ids = []
        self.last_rewards = {}
        self.showdown_results = []

        self.action_history = []

    #start a new hand
    def reset(self, seed: Optional[int] = None, rotate_dealer: bool = True) -> Dict[str, Any]:
        if seed is not None:
            self.rng.seed(seed)

        self.deck = create_deck()
        self.rng.shuffle(self.deck)

        self.community_cards = []
        self.pot = 0
        self.current_bet = 0
        self.round_name = "preflop"

        self.raise_count = 0
        self.acted_this_round = set()

        self.done = False
        self.terminal_reason = None
        self.winner_ids = []
        self.last_rewards = {
            i: 0.0 for i in range(self.config.n_players)
        }
        self.showdown_results = []

        self.action_history = []

        if rotate_dealer:
            self.dealer_button = (self.dealer_button + 1) % self.config.n_players
        else:
            self.dealer_button = 0

        self.players = [
            PlayerState(
                player_id=i,
                stack=self.config.starting_stack
            )
            for i in range(self.config.n_players)
        ]

        self._deal_hole_cards()
        self._post_blinds()

        self.current_player = self._next_active_player(
            start_pos=self._first_to_act_preflop()
        )

        return self.get_observation(self.current_player)

    # execute one action for the current player
    def step(self, action: int):
        if self.done:
            raise RuntimeError("The hand is already finished. Please call reset().")

        if self.current_player is None:
            raise RuntimeError("No current player is available.")

        if action not in ACTIONS:
            raise ValueError(f"Invalid action: {action}. Valid actions are {list(ACTIONS.keys())}.")

        player_id = self.current_player
        player = self.players[player_id]

        info = {
            "player_id": player_id,
            "raw_action": action,
            "raw_action_name": ACTIONS[action],
            "adjusted_action": None,
            "invalid_action_adjusted": False
        }

        legal_actions = self.get_legal_actions(player_id)

        # If the chosen action is illegal, map it to the closest safe action
        if action not in legal_actions:
            info["invalid_action_adjusted"] = True

            # Folding when no bet is faced is treated as check
            if action==0:
                action=1

            # Raising after the raise cap is reached is treated as call/check
            elif action==2:
                action=1

        info["adjusted_action"] = action
        info["adjusted_action_name"] = ACTIONS[action]

        if action==0:
            self._apply_fold(player_id)

        elif action==1:
            self._apply_check_call(player_id)

        elif action==2:
            self._apply_bet_raise(player_id)

        # if only one player remains, the hand ends immediately
        if self._count_active_players()==1:
            self._finish_by_fold()
            return None, self.last_rewards, self.done, info

        # check whether the current betting round is complete
        if self._is_betting_round_complete():
            self._advance_round()

            if self.done:
                return None, self.last_rewards, self.done, info

            next_obs = self.get_observation(self.current_player)
            return next_obs, self.last_rewards, self.done, info

        # Otherwise, move to the next active player
        self.current_player = self._next_active_player(
            start_pos=(player_id+1) % self.config.n_players
        )

        next_obs = self.get_observation(self.current_player)
        return next_obs, self.last_rewards, self.done, info

    def get_legal_actions(self, player_id: int) -> List[int]:
        player = self.players[player_id]

        if player.folded or player.all_in or self.done:
            return []

        to_call = max(0, self.current_bet - player.current_bet)
        legal_actions = []

        if to_call > 0:
            # When facing a bet, the player can fold or call
            legal_actions.extend([0, 1])

            # player can raise only if the raise cap has not been reached
            if self.raise_count<self.config.max_raises_per_round:
                legal_actions.append(2)

        else:
            legal_actions.append(1)
            if self.raise_count<self.config.max_raises_per_round:
                legal_actions.append(2)

        return legal_actions

    def _apply_fold(self, player_id: int):
        player = self.players[player_id]
        player.folded = True
        self.acted_this_round.add(player_id)

        self.action_history.append({
            "round": self.round_name,
            "player": player_id,
            "action": "fold",
            "amount": 0
        })

    def _apply_check_call(self, player_id: int):
        player = self.players[player_id]
        to_call = max(0, self.current_bet - player.current_bet)

        if to_call>0:
            paid = self._pay_chips(player_id, to_call)
            action_name = "call"
        else:
            paid=0
            action_name = "check"

        self.acted_this_round.add(player_id)

        self.action_history.append({
            "round": self.round_name,
            "player": player_id,
            "action": action_name,
            "amount": paid
        })

    def _apply_bet_raise(self, player_id: int):
        player = self.players[player_id]

        old_current_bet = self.current_bet
        target_bet = self.current_bet + self.config.fixed_bet
        amount_to_pay = max(0, target_bet - player.current_bet)

        paid = self._pay_chips(player_id, amount_to_pay)

        # if the player successfully increases the current bet, this is treated as a bet or raise
        if player.current_bet > old_current_bet:
            self.current_bet = player.current_bet
            self.raise_count+=1

            # after a bet or raise, all previous actions are no longer sufficient, other active players need to respond to the new bet
            self.acted_this_round = {player_id}

            if old_current_bet==0:
                action_name = "bet"
            else:
                action_name = "raise"

        else:
            # this situation happens only in rare all-in cases with insufficient chips
            self.acted_this_round.add(player_id)
            action_name = "all_in_call"

        self.action_history.append({
            "round": self.round_name,
            "player": player_id,
            "action": action_name,
            "amount": paid
        })

    #Check whether the current betting round is complete
    def _is_betting_round_complete(self) -> bool:

        actionable_players = [
            p for p in self.players
            if not p.folded and not p.all_in
        ]

        if len(actionable_players)<=1:
            return True

        for p in actionable_players:
            if p.player_id not in self.acted_this_round:
                return False

            if p.current_bet < self.current_bet:
                return False

        return True

    # Move to the next stage of the hand
    def _advance_round(self):
      round_order = ["preflop", "flop", "turn", "river", "showdown"]
      current_index = round_order.index(self.round_name)
      next_round = round_order[current_index + 1]

      # reset per-round betting information
      self.current_bet = 0
      self.raise_count = 0
      self.acted_this_round = set()

      for p in self.players:
          p.current_bet = 0

      if next_round == "flop":
          self.community_cards.extend([self._deal_one_card() for _ in range(3)])
          self.round_name = "flop"
          self.current_player = self._next_active_player(
              start_pos=self._first_to_act_postflop()
          )

      elif next_round == "turn":
          self.community_cards.append(self._deal_one_card())
          self.round_name = "turn"
          self.current_player = self._next_active_player(
              start_pos=self._first_to_act_postflop()
          )

      elif next_round == "river":
          self.community_cards.append(self._deal_one_card())
          self.round_name = "river"
          self.current_player = self._next_active_player(
              start_pos=self._first_to_act_postflop()
          )

      elif next_round == "showdown":
          self.round_name = "showdown"
          self.current_player = None
          self._finish_by_showdown()

    def _finish_by_fold(self):
        active_players = [
            p for p in self.players
            if not p.folded
        ]

        if len(active_players) != 1:
            raise RuntimeError("finish_by_fold called, but more than one player is still active.")

        winner = active_players[0]
        winner.stack += self.pot

        self.winner_ids = [winner.player_id]
        self.terminal_reason = "all_others_folded"
        self.pot = 0
        self.done = True
        self.current_player = None

        self.last_rewards = {
            p.player_id: p.stack - self.config.starting_stack
            for p in self.players
        }

    def _finish_by_showdown(self):
        active_players = [
            p for p in self.players
            if not p.folded
        ]

        if len(active_players) == 0:
            raise RuntimeError("No active players at showdown.")

        showdown_results = []

        for player in active_players:
            seven_cards = player.hole_cards + self.community_cards
            best_score, best_cards = evaluate_best_hand(seven_cards)

            showdown_results.append({
                "player_id": player.player_id,
                "score": best_score,
                "hand_name": hand_score_to_name(best_score),
                "best_cards": best_cards
            })

        best_score = max(result["score"] for result in showdown_results)

        winners = [
            result["player_id"]
            for result in showdown_results
            if result["score"] == best_score
        ]

        self._award_pot_to_winners(winners)

        self.winner_ids = winners
        self.terminal_reason = "showdown"
        self.done = True
        self.current_player = None

        self.showdown_results = showdown_results

        self.last_rewards = {
            p.player_id: p.stack - self.config.starting_stack
            for p in self.players
        }

    # split the pot among winners
    def _award_pot_to_winners(self, winner_ids: List[int]):
        if len(winner_ids) == 0:
            raise ValueError("winner_ids cannot be empty.")

        pot_share = self.pot // len(winner_ids)
        remainder = self.pot % len(winner_ids)

        for i, winner_id in enumerate(winner_ids):
            self.players[winner_id].stack += pot_share

            # Give any leftover chip to the earliest winner in the list
            if i < remainder:
                self.players[winner_id].stack += 1

        self.pot = 0

    def _count_active_players(self) -> int:
        return sum(not p.folded for p in self.players)

    def _deal_one_card(self) -> Card:
        if not self.deck:
            raise RuntimeError("Deck is empty.")
        return self.deck.pop()

    # deal two private hole cards to each player. (The dealing order starts from the player to the left of the dealer
    def _deal_hole_cards(self):
        start_pos = (self.dealer_button+1) % self.config.n_players

        for _ in range(2):
            for offset in range(self.config.n_players):
                player_pos = (start_pos + offset) % self.config.n_players
                self.players[player_pos].hole_cards.append(self._deal_one_card())

    def _small_blind_pos(self) -> int:
        if self.config.n_players == 2:
            return self.dealer_button
        return (self.dealer_button + 1) % self.config.n_players

    def _big_blind_pos(self) -> int:
        if self.config.n_players == 2:
            return (self.dealer_button + 1) % self.config.n_players
        return (self.dealer_button + 2) % self.config.n_players

    def _pay_chips(self, player_id: int, amount: int):
        player = self.players[player_id]
        actual_payment = min(player.stack, amount)

        player.stack -= actual_payment
        player.current_bet += actual_payment
        player.total_contribution += actual_payment
        self.pot += actual_payment

        if player.stack==0:
            player.all_in = True

        return actual_payment

    def _post_blinds(self):
        sb_pos = self._small_blind_pos()
        bb_pos = self._big_blind_pos()

        sb_paid = self._pay_chips(sb_pos, self.config.small_blind)
        bb_paid = self._pay_chips(bb_pos, self.config.big_blind)

        self.current_bet = max(sb_paid, bb_paid)

        self.action_history.append({
            "round": self.round_name,
            "player": sb_pos,
            "action": "small_blind",
            "amount": sb_paid
        })

        self.action_history.append({
            "round": self.round_name,
            "player": bb_pos,
            "action": "big_blind",
            "amount": bb_paid
        })

    def _first_to_act_preflop(self) -> int:
        bb_pos = self._big_blind_pos()
        return (bb_pos + 1) % self.config.n_players

    def _first_to_act_postflop(self) -> int:
        return (self.dealer_button + 1) % self.config.n_players

    def _next_active_player(self, start_pos: int) -> Optional[int]:
        for offset in range(self.config.n_players):
            pos = (start_pos + offset) % self.config.n_players
            player = self.players[pos]

            if not player.folded and not player.all_in:
                return pos

        return None

    # return the observation available to a given player, but other players' hole cards are not revealed
    def get_observation(self, player_id: int) -> Dict[str, Any]:
        player = self.players[player_id]
        to_call = max(0, self.current_bet - player.current_bet)

        obs = {
            "player_id": player_id,
            "round": self.round_name,

            "hole_cards": tuple(player.hole_cards),
            "community_cards": tuple(self.community_cards),

            "pot": self.pot,
            "current_bet": self.current_bet,
            "to_call": to_call,

            "stacks": tuple(p.stack for p in self.players),
            "current_bets": tuple(p.current_bet for p in self.players),
            "folded": tuple(p.folded for p in self.players),
            "all_in": tuple(p.all_in for p in self.players),

            "dealer_button": self.dealer_button,
            "small_blind_pos": self._small_blind_pos(),
            "big_blind_pos": self._big_blind_pos(),
            "current_player": self.current_player,

            "legal_actions": tuple(self.get_legal_actions(player_id)),

            "done": self.done
        }

        return obs

## 2. Rule-based Opponents
The following rule-based opponents provide different playing styles for training and evaluation. They make the environment less deterministic and create a stronger benchmark than a purely random opponent.


### 2.1 Base agent and legal-action handling
The base class standardizes the `act` interface, while `safe_action` maps the intended actions to legal actions if needed.


In [ ]:
class BaseAgent:

    def __init__(self, name: str):
        self.name = name

    def act(self, obs):
        raise NotImplementedError

#Convert an intended action into a legal action if necessary.
def safe_action(action, legal_actions):
    if action in legal_actions:
        return action

    # if folding is selected but not legal, use check/call instead
    if action == 0 and 1 in legal_actions:
        return 1

    # if bet/raise is selected but not legal, use check/call instead
    if action == 2 and 1 in legal_actions:
        return 1

    # choose the first legal action
    return legal_actions[0]

### 2.2 Simple hand-strength estimator
This heuristic compresses private and community cards into a coarse strength score used by rule-based opponents and the adaptive agent.


In [ ]:
def estimate_simple_hand_strength(hole_cards, community_cards):
    cards = list(hole_cards) + list(community_cards)
    if len(community_cards)== 0:
        ranks = sorted([card[0] for card in hole_cards], reverse=True)
        r1, r2 = ranks

        is_pair = r1 ==r2
        high_card = r1

        if is_pair and r1 >=10:
            return 4
        if is_pair:
            return 3
        if r1 == 14 and r2 >=10:
            return 4
        if r1 >= 13 and r2 >=10:
            return 3
        if high_card >= 12:
            return 2
        if high_card >= 10:
            return 1
        return 0

    if len(cards) >= 5:
        best_score, _ = evaluate_best_hand(cards)
        hand_category = best_score[0]

        if hand_category>=6:
            return 4       # full house or better
        elif hand_category >=4:
            return 3       # straight or flush
        elif hand_category >=2:
            return 2       # two pair or three of a kind
        elif hand_category ==1:
            return 1       # one pair
        else:
            return 0       # high card only

    # if there are fewer than 5 cards after flop/turn, use rank-count logic
    ranks = [card[0] for card in cards]
    rank_counts = Counter(ranks)
    max_count = max(rank_counts.values())

    if max_count >=3:
        return 3
    if max_count ==2:
        return 1

    high_card = max(ranks)
    if high_card >=13:
        return 1

    return 0

### 2.3 Random opponent


In [ ]:
class RandomAgent(BaseAgent):
    def __init__(self):
        super().__init__(name="RandomAgent")

    def act(self, obs):
        legal_actions = list(obs["legal_actions"])
        return random.choice(legal_actions)

### 2.4 Passive opponent


In [ ]:
# passive agent prefers checking and calling and it rarely raises, even with strong hands
class PassiveAgent(BaseAgent):

    def __init__(self):
        super().__init__(name="PassiveAgent")

    def act(self, obs):
        legal_actions = list(obs["legal_actions"])
        strength = estimate_simple_hand_strength(
            obs["hole_cards"],
            obs["community_cards"]
        )

        to_call = obs["to_call"]

        # If facing a bet with a very weak hand, sometimes fold.
        if to_call > 0 and strength == 0:
            intended_action = random.choices(
                population=[0, 1],
                weights=[0.6, 0.4]
            )[0]
            return safe_action(intended_action, legal_actions)

        # With strong hands, occasionally bet/raise.
        if strength >= 3 and 2 in legal_actions:
            intended_action = random.choices(
                population=[1, 2],
                weights=[0.75, 0.25]
            )[0]
            return safe_action(intended_action, legal_actions)

        # Default behavior: check/call.
        return safe_action(1, legal_actions)

### 2.5 Aggressive opponent


In [ ]:
# aggressive agent will frequently bets and raises
class AggressiveAgent(BaseAgent):
    def __init__(self):
        super().__init__(name="AggressiveAgent")

    def act(self, obs):
        legal_actions = list(obs["legal_actions"])
        strength = estimate_simple_hand_strength(
            obs["hole_cards"],
            obs["community_cards"]
        )

        to_call = obs["to_call"]

        # strong hands: raise very often
        if strength >= 3 and 2 in legal_actions:
            intended_action = random.choices(
                population=[1, 2],
                weights=[0.25, 0.75]
            )[0]
            return safe_action(intended_action, legal_actions)

        # medium hands: raise sometimes
        if strength == 2 and 2 in legal_actions:
            intended_action = random.choices(
                population=[1, 2],
                weights=[0.55, 0.45]
            )[0]
            return safe_action(intended_action, legal_actions)

        # weak hands facing a bet: sometimes fold, but still calls often
        if to_call > 0 and strength <=1:
            intended_action = random.choices(
                population=[0, 1],
                weights=[0.25, 0.75]
            )[0]
            return safe_action(intended_action, legal_actions)

        # if no bet is faced, sometimes bet with weak hands
        if to_call == 0 and 2 in legal_actions:
            intended_action = random.choices(
                population=[1, 2],
                weights=[0.65, 0.35]
            )[0]
            return safe_action(intended_action, legal_actions)

        return safe_action(1, legal_actions)

### 2.6 Bluff-heavy opponent


In [ ]:
# a bluff-heavy agent will sometimes raises with weak hands. This agent is designed to make the environment less predictable
class BluffHeavyAgent(BaseAgent):
    def __init__(self):
        super().__init__(name="BluffHeavyAgent")

    def act(self, obs):
        legal_actions = list(obs["legal_actions"])
        strength = estimate_simple_hand_strength(
            obs["hole_cards"],
            obs["community_cards"]
        )

        to_call = obs["to_call"]

        # very strong hands: often raise for value
        if strength >= 3 and 2 in legal_actions:
            intended_action = random.choices(
                population=[1, 2],
                weights=[0.35, 0.65]
            )[0]
            return safe_action(intended_action, legal_actions)

        # weak or medium hands: still bluff with some probability
        if 2 in legal_actions:
            if strength <= 1:
                intended_action = random.choices(
                    population=[0, 1, 2] if to_call > 0 else [1, 2],
                    weights=[0.25, 0.35, 0.40] if to_call > 0 else [0.55, 0.45]
                )[0]
                return safe_action(intended_action, legal_actions)

            if strength == 2:
                intended_action = random.choices(
                    population=[1, 2],
                    weights=[0.5, 0.5]
                )[0]
                return safe_action(intended_action, legal_actions)

        # if raising is not available, decide between fold and call
        if to_call > 0 and strength == 0:
            intended_action = random.choices(
                population=[0, 1],
                weights=[0.4, 0.6]
            )[0]
            return safe_action(intended_action, legal_actions)

        return safe_action(1, legal_actions)

### 2.7 Tight opponent


In [ ]:
# A tight agent that folds weak hands and only plays stronger hands.
class TightAgent(BaseAgent):
    def __init__(self):
        super().__init__(name="TightAgent")

    def act(self, obs):
        legal_actions = list(obs["legal_actions"])
        strength = estimate_simple_hand_strength(
            obs["hole_cards"],
            obs["community_cards"]
        )

        to_call = obs["to_call"]

        # fold very weak hands when facing a bet
        if to_call > 0 and strength <=1:
            intended_action = random.choices(
                population=[0, 1],
                weights=[0.75, 0.25]
            )[0]
            return safe_action(intended_action, legal_actions)

        # strong hands: bet or raise
        if strength >= 3 and 2 in legal_actions:
            intended_action = random.choices(
                population=[1, 2],
                weights=[0.35, 0.65]
            )[0]
            return safe_action(intended_action, legal_actions)

        # medium hands: mostly check/call
        if strength==2:
            return safe_action(1, legal_actions)

        # weak hands with no bet: check
        return safe_action(1, legal_actions)

## 3. Q-learning Agent


### 3.1 State encoding
The original poker observation is converted into a discrete tuple so that a tabular Q-learning method can be applied.


In [ ]:
ROUND_TO_ID = {
    "preflop": 0,
    "flop": 1,
    "turn": 2,
    "river": 3,
    "showdown": 4
}

# Convert the required call amount into a discrete bucket:
# 0 = no call needed
# 1 = small call
# 2 = medium call
# 3 = large call
def bucket_to_call(to_call, fixed_bet):
    if to_call == 0:
        return 0

    elif to_call <= fixed_bet:
        return 1

    elif to_call <= 2 * fixed_bet:
        return 2

    else:
        return 3

# 0 = very small pot
# 1 = small pot
# 2 = medium pot
# 3 = large pot
# 4 = very large pot
def bucket_pot_size(pot, big_blind):
    if pot <= 2*big_blind:
        return 0

    elif pot <= 5*big_blind:
        return 1

    elif pot <= 10*big_blind:
        return 2

    elif pot <= 20*big_blind:
        return 3

    else:
        return 4

# 0 = short stack
# 1 = medium-short stack
# 2 = medium stack
# 3 = healthy stack
# 4 = deep stack
def bucket_stack_size(stack, starting_stack):
    ratio = stack / starting_stack

    if ratio <= 0.25:
        return 0

    elif ratio <= 0.5:
        return 1

    elif ratio <= 0.75:
        return 2

    elif ratio <= 1:
        return 3

    else:
        return 4


# 0 = dealer button
# 1 = small blind
# 2 = big blind
# 3 = early position
# 4 = middle position
# 5 = late position
def encode_position(player_id, dealer_button, small_blind_pos, big_blind_pos, n_players):

    if player_id == dealer_button:
        return 0

    if player_id == small_blind_pos:
        return 1

    if player_id == big_blind_pos:
        return 2

    relative_pos = (player_id - dealer_button) % n_players

    if relative_pos <= n_players/3:
        return 3

    elif relative_pos <= 2*n_players/3:
        return 4

    else:
        return 5


def count_active_players_from_obs(obs):
    return sum(not folded for folded in obs["folded"])


# 0 = no active bet
# 1 = current player has already matched the bet
# 2 = facing a normal bet
# 3 = facing a larger betting pressure
def bucket_betting_pressure(obs, config):
    current_bet = obs["current_bet"]
    to_call = obs["to_call"]

    if current_bet == 0:
        return 0

    if to_call == 0:
        return 1

    if to_call <= config.fixed_bet:
        return 2

    return 3


def encode_state(obs, config):
    player_id = obs["player_id"]

    round_id = ROUND_TO_ID[obs["round"]]

    hand_strength = estimate_simple_hand_strength(
        obs["hole_cards"],
        obs["community_cards"]
    )

    draw_strength = estimate_draw_strength(
        obs["hole_cards"],
        obs["community_cards"]
    )

    to_call_bucket = bucket_to_call(
        obs["to_call"],
        config.fixed_bet
    )

    pot_bucket = bucket_pot_size(
        obs["pot"],
        config.big_blind
    )

    player_stack = obs["stacks"][player_id]

    stack_bucket = bucket_stack_size(
        player_stack,
        config.starting_stack
    )

    n_players = len(obs["stacks"])

    position_bucket = encode_position(
        player_id=player_id,
        dealer_button=obs["dealer_button"],
        small_blind_pos=obs["small_blind_pos"],
        big_blind_pos=obs["big_blind_pos"],
        n_players=n_players
    )

    active_players_count = count_active_players_from_obs(obs)

    legal_actions = obs["legal_actions"]

    raise_available = int(2 in legal_actions)

    betting_pressure_bucket = bucket_betting_pressure(
        obs=obs,
        config=config
    )

    state = (
        round_id,
        hand_strength,
        draw_strength,
        to_call_bucket,
        pot_bucket,
        stack_bucket,
        position_bucket,
        active_players_count,
        raise_available,
        betting_pressure_bucket
    )

    return state

### 3.2 Tabular Q-learning implementation
The Q-learning agent uses the epsilon-greedy exploration, legal-action masking, and standard temporal-difference updates.


In [ ]:
class QLearningAgent(BaseAgent):

    def __init__(
        self,
        name="QLearningAgent",
        n_actions=3,
        alpha=0.03,
        gamma=0.95,
        epsilon=1,
        epsilon_min=0.05,
        epsilon_decay=0.995,
        config=None
    ):
        super().__init__(name=name)

        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma

        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay

        self.config = config

        # Q-table:
        # key = encoded state
        # value = Q-values for all actions
        self.Q = defaultdict(lambda: np.zeros(self.n_actions))

    #Select an action using epsilon-greedy policy.
    def select_action(self, state, legal_actions, training=True):

        legal_actions = list(legal_actions)

        if len(legal_actions) == 0:
            raise ValueError("No legal actions available.")

        # exploration
        if training and random.random() < self.epsilon:
            return random.choice(legal_actions)

        # exploitation with legal-action masking
        q_values = self.Q[state]

        legal_q_values = {
            action: q_values[action]
            for action in legal_actions
        }

        max_q = max(legal_q_values.values())

        # random tie-breaking among equally good actions
        best_actions = [
            action for action, q in legal_q_values.items()
            if q == max_q
        ]

        return random.choice(best_actions)

    def act(self, obs, config=None):
        if config is None:
            config = self.config

        if config is None:
            raise ValueError(
                "config must be provided either when initializing "
                "QLearningAgent or when calling act(obs, config)."
            )

        state = encode_state(obs, config)
        legal_actions = obs["legal_actions"]

        return self.select_action(
            state=state,
            legal_actions=legal_actions,
            training=False
        )

    #update Q-value using the standard Q-learning formula: Q(s, a) <- Q(s, a) + alpha * [target - Q(s, a)]
    def update(self, state, action, reward, next_state=None, next_legal_actions=None, done=False):
        current_q = self.Q[state][action]

        if done:
            target = reward

        else:
            if next_state is None:
                raise ValueError("next_state cannot be None when done=False.")

            if next_legal_actions is None or len(next_legal_actions) == 0:
                next_max_q = 0.0

            else:
                next_q_values = self.Q[next_state]
                next_max_q = max(next_q_values[a] for a in next_legal_actions)

            target = reward + self.gamma * next_max_q

        self.Q[state][action] = current_q + self.alpha * (target - current_q)

    def decay_epsilon(self):
        self.epsilon = max(
            self.epsilon_min,
            self.epsilon * self.epsilon_decay
        )

    def get_q_values(self, state):
        return self.Q[state]

    def set_epsilon(self, epsilon):
        self.epsilon = epsilon

## 4. Training against a fixed mixed opponent pool
Player 0 is trained as the Q-learning agent for 20,000 hands against a fixed mixture of passive, aggressive, and bluff-heavy opponents.


In [ ]:
def moving_average(values, window=1000):
    if len(values) < window:
        return values

    values = np.array(values, dtype=float)
    return np.convolve(values, np.ones(window) / window, mode="valid")

def train_one_hand_q_learning(
    env,
    q_agent,
    opponent_agents,
    rl_player_id=0,
    seed=None
):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    env.reset(seed=seed, rotate_dealer=True)

    pending_state = None
    pending_action = None

    while not env.done:
        current_player = env.current_player
        obs = env.get_observation(current_player)

        if current_player == rl_player_id:
            current_state = encode_state(obs, env.config)
            legal_actions = obs["legal_actions"]

            # if the RL agent has acted before, update the previous transition
            # intermediate reward is set to 0 because poker reward is mainly terminal
            if pending_state is not None and pending_action is not None:
                q_agent.update(
                    state=pending_state,
                    action=pending_action,
                    reward=0.0,
                    next_state=current_state,
                    next_legal_actions=legal_actions,
                    done=False
                )

            action = q_agent.select_action(
                state=current_state,
                legal_actions=legal_actions,
                training=True
            )

            next_obs, rewards, done, info = env.step(action)

            #if the hand ends immediately after the RL action, update with final reward.
            if env.done:
                final_reward = env.last_rewards[rl_player_id]

                q_agent.update(
                    state=current_state,
                    action=action,
                    reward=final_reward,
                    next_state=None,
                    next_legal_actions=None,
                    done=True
                )

                pending_state = None
                pending_action = None

            else:
                #store this RL decision until the next RL turn or terminal state.
                pending_state = current_state
                pending_action = action

        #current player is a rule-based opponent
        else:
            opponent = opponent_agents[current_player]
            action = opponent.act(obs)

            next_obs, rewards, done, info = env.step(action)

            #if the hand ends after an opponent action, update the last RL action.
            if env.done and pending_state is not None and pending_action is not None:
                final_reward = env.last_rewards[rl_player_id]

                q_agent.update(
                    state=pending_state,
                    action=pending_action,
                    reward=final_reward,
                    next_state=None,
                    next_legal_actions=None,
                    done=True
                )

                pending_state = None
                pending_action = None

    final_reward = env.last_rewards[rl_player_id]
    is_winner = int(rl_player_id in env.winner_ids)

    result = {
        "reward": final_reward,
        "is_winner": is_winner,
        "winner_ids": env.winner_ids,
        "terminal_reason": env.terminal_reason,
        "action_history": list(env.action_history),
        "final_rewards": env.last_rewards.copy()
    }

    return result

def train_q_learning_agent(
    n_episodes=20000,
    seed_start=0,
    verbose_every=500
):
    config = PokerConfig(
        n_players=4,
        starting_stack=100,
        small_blind=1,
        big_blind=2,
        fixed_bet=2,
        max_raises_per_round=2
    )

    env = FixedLimitTexasHoldemEnv(config)

    q_agent = QLearningAgent(
        name="QLearningAgent",
        n_actions=3,
        alpha=0.03,
        gamma=0.95,
        epsilon=1,
        epsilon_min=0.05,
        epsilon_decay=0.9997
    )

    opponent_agents = {
        1: PassiveAgent(),
        2: AggressiveAgent(),
        3: BluffHeavyAgent()
    }

    rewards = []
    wins = []
    epsilons = []
    terminal_reasons = []
    q_table_sizes = []

    for episode in range(n_episodes):
        result = train_one_hand_q_learning(
            env=env,
            q_agent=q_agent,
            opponent_agents=opponent_agents,
            rl_player_id=0,
            seed=seed_start + episode
        )

        rewards.append(result["reward"])
        wins.append(result["is_winner"])
        epsilons.append(q_agent.epsilon)
        terminal_reasons.append(result["terminal_reason"])
        q_table_sizes.append(len(q_agent.Q))

        q_agent.decay_epsilon()

        if verbose_every is not None and (episode + 1) % verbose_every == 0:
            recent_reward = np.mean(rewards[-verbose_every:])
            recent_win_rate = np.mean(wins[-verbose_every:])

            print(
                f"Episode {episode + 1}/{n_episodes} | "
                f"Recent avg reward: {recent_reward:.3f} | "
                f"Recent win rate: {recent_win_rate:.3f} | "
                f"Epsilon: {q_agent.epsilon:.3f} | "
                f"Q-table size: {len(q_agent.Q)}"
            )

    history = {
        "rewards": rewards,
        "wins": wins,
        "epsilons": epsilons,
        "terminal_reasons": terminal_reasons,
        "q_table_sizes": q_table_sizes
    }

    return q_agent, history

In [ ]:
q_agent, train_history = train_q_learning_agent(
    n_episodes=20000,
    seed_start=1000,
    verbose_every=500
)

NameError: name 'evaluate_best_hand' is not defined

In [ ]:
rewards = train_history["rewards"]

plt.figure(figsize=(10, 5))
plt.plot(moving_average(rewards, window=1000), label="1000-hand moving average")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Q-learning Training Reward")
plt.legend()
plt.show()

In [ ]:
wins = train_history["wins"]

plt.figure(figsize=(10, 5))
plt.plot(moving_average(wins, window=1000), label="1000-hand moving win rate")
plt.xlabel("Episode")
plt.ylabel("Win Rate")
plt.title("Q-learning Training Win Rate")
plt.legend()
plt.show()

In [ ]:
epsilons = train_history["epsilons"]

plt.figure(figsize=(10,5))
plt.plot(epsilons)
plt.xlabel("Episode")
plt.ylabel("Epsilon")
plt.title("Epsilon Decay During Training")
plt.show()

In [ ]:
q_table_sizes = train_history["q_table_sizes"]

plt.figure(figsize=(10,5))
plt.plot(q_table_sizes)
plt.xlabel("Episode")
plt.ylabel("Number of Visited States")
plt.title("Q-table Size During Training")
plt.show()

### 4.1 Baseline comparison
The trained Q-learning agent is compared with the rule-based agents under the same fixed opponent composition.


In [ ]:
def evaluate_candidate_agent(
    candidate_agent,
    n_hands=1000,
    seed_start=30000,
    candidate_player_id=0,
    use_q_policy=False
):
    config = PokerConfig(
        n_players=4,
        starting_stack=100,
        small_blind=1,
        big_blind=2,
        fixed_bet=2,
        max_raises_per_round=2
    )

    env = FixedLimitTexasHoldemEnv(config)

    opponent_agents = {
        1: PassiveAgent(),
        2: AggressiveAgent(),
        3: BluffHeavyAgent()
    }

    old_epsilon = None

    if use_q_policy:
        old_epsilon = candidate_agent.epsilon
        candidate_agent.set_epsilon(0.0)

    rewards = []
    wins = []
    terminal_reasons = []

    for i in range(n_hands):
        # set random seed
        random.seed(seed_start + i)

        env.reset(seed=seed_start + i, rotate_dealer=True)

        while not env.done:
            current_player = env.current_player
            obs = env.get_observation(current_player)

            if current_player == candidate_player_id:
                if use_q_policy:
                    state = encode_state(obs, env.config)
                    action = candidate_agent.select_action(
                        state=state,
                        legal_actions=obs["legal_actions"],
                        training=False
                    )
                else:
                    action = candidate_agent.act(obs)

            else:
                opponent = opponent_agents[current_player]
                action = opponent.act(obs)

            env.step(action)

        rewards.append(env.last_rewards[candidate_player_id])
        wins.append(int(candidate_player_id in env.winner_ids))
        terminal_reasons.append(env.terminal_reason)

    if use_q_policy:
        candidate_agent.set_epsilon(old_epsilon)

    summary = {
        "agent_name": candidate_agent.name,
        "avg_reward": float(np.mean(rewards)),
        "reward_std": float(np.std(rewards)),
        "win_rate": float(np.mean(wins)),
        "terminal_reasons": Counter(terminal_reasons),
        "all_rewards": rewards,
        "all_wins": wins
    }

    return summary

In [ ]:
baseline_agents = [
    RandomAgent(),
    PassiveAgent(),
    AggressiveAgent(),
    BluffHeavyAgent(),
    TightAgent()
]

baseline_results = []

for agent in baseline_agents:
    result = evaluate_candidate_agent(
        candidate_agent=agent,
        n_hands=1000,
        seed_start=40000,
        candidate_player_id=0,
        use_q_policy=False
    )
    baseline_results.append(result)

q_result = evaluate_candidate_agent(
    candidate_agent=q_agent,
    n_hands=1000,
    seed_start=40000,
    candidate_player_id=0,
    use_q_policy=True
)

baseline_results.append(q_result)

comparison_rows = []

for result in baseline_results:
    comparison_rows.append({
        "Agent": result["agent_name"],
        "Average Reward": result["avg_reward"],
        "Reward Std": result["reward_std"],
        "Win Rate": result["win_rate"],
        "Showdown Count": result["terminal_reasons"].get("showdown", 0),
        "Fold Finish Count": result["terminal_reasons"].get("all_others_folded", 0)
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df.sort_values(
    by="Average Reward",
    ascending=False
).reset_index(drop=True)

comparison_df

## 5. Evaluation across different opponent pools


### 5.1 Opponent-pool definition
Five opponents settings are used: mostly passive, mostly aggressive, mostly bluff-heavy, mostly random, and mixed.


In [ ]:
#Player 0 is always the evaluated Q-learning agent while players 1, 2, and 3 are rule-based opponents.
def create_opponent_pool(pool_type):
    if pool_type == "mostly_passive":
        return {
            1: PassiveAgent(),
            2: PassiveAgent(),
            3: TightAgent()
        }

    elif pool_type == "mostly_aggressive":
        return {
            1: AggressiveAgent(),
            2: AggressiveAgent(),
            3: BluffHeavyAgent()
        }

    elif pool_type == "mostly_bluff":
        return {
            1: BluffHeavyAgent(),
            2: BluffHeavyAgent(),
            3: AggressiveAgent()
        }

    elif pool_type == "mostly_random":
        return {
            1: RandomAgent(),
            2: RandomAgent(),
            3: BluffHeavyAgent()
        }

    elif pool_type == "mixed":
        return {
            1: PassiveAgent(),
            2: AggressiveAgent(),
            3: BluffHeavyAgent()
        }

    else:
        raise ValueError(f"Unknown pool_type: {pool_type}")

### 5.2 Fixed Q-agent evaluation
The trained Q-learning agent is evaluated separately in each opponent pool to test whether its learned policy generalizes across different playing styles.


In [ ]:
def evaluate_q_agent_in_opponent_pool(
    q_agent,
    pool_type,
    n_hands=1000,
    seed_start=60000,
    rl_player_id=0
):

    config = PokerConfig(
        n_players=4,
        starting_stack=100,
        small_blind=1,
        big_blind=2,
        fixed_bet=2,
        max_raises_per_round=2
    )

    env = FixedLimitTexasHoldemEnv(config)

    opponent_agents = create_opponent_pool(pool_type)

    old_epsilon = q_agent.epsilon
    q_agent.set_epsilon(0.0)

    rewards = []
    wins = []
    terminal_reasons = []

    for i in range(n_hands):
        random.seed(seed_start + i)
        env.reset(seed=seed_start + i, rotate_dealer=True)

        while not env.done:
            current_player = env.current_player
            obs = env.get_observation(current_player)

            if current_player == rl_player_id:
                state = encode_state(obs, env.config)

                action = q_agent.select_action(
                    state=state,
                    legal_actions=obs["legal_actions"],
                    training=False
                )

            else:
                opponent = opponent_agents[current_player]
                action = opponent.act(obs)

            env.step(action)

        rewards.append(env.last_rewards[rl_player_id])
        wins.append(int(rl_player_id in env.winner_ids))
        terminal_reasons.append(env.terminal_reason)

    q_agent.set_epsilon(old_epsilon)

    summary = {
        "pool_type": pool_type,
        "avg_reward": float(np.mean(rewards)),
        "reward_std": float(np.std(rewards)),
        "win_rate": float(np.mean(wins)),
        "showdown_count": Counter(terminal_reasons).get("showdown", 0),
        "fold_finish_count": Counter(terminal_reasons).get("all_others_folded", 0),
        "all_rewards": rewards,
        "all_wins": wins,
        "terminal_reasons": terminal_reasons
    }

    return summary

pool_types = [
    "mostly_passive",
    "mostly_aggressive",
    "mostly_bluff",
    "mostly_random",
    "mixed"
]

pool_results = []

for pool_type in pool_types:
    result = evaluate_q_agent_in_opponent_pool(
        q_agent=q_agent,
        pool_type=pool_type,
        n_hands=1000,
        seed_start=60000
    )

    pool_results.append(result)

pool_comparison_rows = []

for result in pool_results:
    pool_comparison_rows.append({
        "Opponent Pool": result["pool_type"],
        "Average Reward": result["avg_reward"],
        "Reward Std": result["reward_std"],
        "Win Rate": result["win_rate"],
        "Showdown Count": result["showdown_count"],
        "Fold Finish Count": result["fold_finish_count"]
    })

pool_comparison_df = pd.DataFrame(pool_comparison_rows)

pool_comparison_df = pool_comparison_df.sort_values(
    by="Average Reward",
    ascending=False
).reset_index(drop=True)

pool_comparison_df


## 6. Training under a time-varying opponents pool
This experiment trains another Q-learning agent for 20,000 episodes while the opponent pool changes every 5,000 episodes. The phase order is taht: mixed, mostly passive, mostly aggressive, and mostly bluff-heavy.


In [ ]:
N_TRAIN_EPISODES = 20000
PHASE_LENGTH = N_TRAIN_EPISODES//4

def get_phase_by_episode(episode):
    if episode < PHASE_LENGTH:
        return "mixed"
    elif episode < 2*PHASE_LENGTH:
        return "mostly_passive"
    elif episode < 3*PHASE_LENGTH:
        return "mostly_aggressive"
    else:
        return "mostly_bluff"

In [ ]:
def train_q_learning_time_varying_pool(
    n_episodes=20000,
    seed_start=70000,
    verbose_every=500
):

    config = PokerConfig(
        n_players=4,
        starting_stack=100,
        small_blind=1,
        big_blind=2,
        fixed_bet=2,
        max_raises_per_round=2
    )

    env = FixedLimitTexasHoldemEnv(config)

    q_agent_tv = QLearningAgent(
        name="QLearningAgent_TimeVarying",
        n_actions=3,
        alpha=0.03,
        gamma=0.95,
        epsilon=1,
        epsilon_min=0.05,
        epsilon_decay=0.9997
    )

    rewards = []
    wins = []
    epsilons = []
    phases = []
    terminal_reasons = []
    q_table_sizes = []

    for episode in range(n_episodes):
        phase = get_phase_by_episode(episode)
        opponent_agents = create_opponent_pool(phase)

        result = train_one_hand_q_learning(
            env=env,
            q_agent=q_agent_tv,
            opponent_agents=opponent_agents,
            rl_player_id=0,
            seed=seed_start + episode
        )

        rewards.append(result["reward"])
        wins.append(result["is_winner"])
        epsilons.append(q_agent_tv.epsilon)
        phases.append(phase)
        terminal_reasons.append(result["terminal_reason"])
        q_table_sizes.append(len(q_agent_tv.Q))

        q_agent_tv.decay_epsilon()

        if verbose_every is not None and (episode+1) % verbose_every==0:
            recent_reward = np.mean(rewards[-verbose_every:])
            recent_win_rate = np.mean(wins[-verbose_every:])

            print(
                f"Episode {episode + 1}/{n_episodes} | "
                f"Phase: {phase} | "
                f"Recent avg reward: {recent_reward:.3f} | "
                f"Recent win rate: {recent_win_rate:.3f} | "
                f"Epsilon: {q_agent_tv.epsilon:.3f} | "
                f"Q-table size: {len(q_agent_tv.Q)}"
            )

    history = {
        "rewards": rewards,
        "wins": wins,
        "epsilons": epsilons,
        "phases": phases,
        "terminal_reasons": terminal_reasons,
        "q_table_sizes": q_table_sizes
    }

    return q_agent_tv, history

q_agent_tv, tv_history = train_q_learning_time_varying_pool(
    n_episodes=20000,
    seed_start=70000,
    verbose_every=500
)

In [ ]:
tv_rewards = tv_history["rewards"]

plt.figure(figsize=(12, 5))
plt.plot(moving_average(tv_rewards, window=1000), label="1000-hand moving average")

plt.axvline(5000, linestyle="--", label="Phase change")
plt.axvline(10000, linestyle="--")
plt.axvline(15000, linestyle="--")

plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Q-learning Reward under Time-varying Opponent Pool")
plt.legend()
plt.show()

In [ ]:
tv_wins = tv_history["wins"]

plt.figure(figsize=(12, 5))
plt.plot(moving_average(tv_wins, window=1000), label="1000-hand moving win rate")

plt.axvline(5000, linestyle="--", label="Phase change")
plt.axvline(10000, linestyle="--")
plt.axvline(15000, linestyle="--")

plt.xlabel("Episode")
plt.ylabel("Win Rate")
plt.title("Q-learning Win Rate under Time-varying Opponent Pool")
plt.legend()
plt.show()

In [ ]:
def compare_fixed_vs_time_varying_agents(
    fixed_agent,
    tv_agent,
    n_hands=1000,
    seed_start=80000
):
    pool_types = [
        "mostly_passive",
        "mostly_aggressive",
        "mostly_bluff",
        "mostly_random",
        "mixed"
    ]

    rows = []

    for pool_type in pool_types:
        fixed_result = evaluate_q_agent_in_opponent_pool(
            q_agent=fixed_agent,
            pool_type=pool_type,
            n_hands=n_hands,
            seed_start=seed_start
        )

        tv_result = evaluate_q_agent_in_opponent_pool(
            q_agent=tv_agent,
            pool_type=pool_type,
            n_hands=n_hands,
            seed_start=seed_start
        )

        rows.append({
            "Opponent Pool": pool_type,
            "Fixed Agent Avg Reward": fixed_result["avg_reward"],
            "TV Agent Avg Reward": tv_result["avg_reward"],
            "Fixed Agent Win Rate": fixed_result["win_rate"],
            "TV Agent Win Rate": tv_result["win_rate"]
        })

    return pd.DataFrame(rows)

fixed_vs_tv_df = compare_fixed_vs_time_varying_agents(
    fixed_agent=q_agent,
    tv_agent=q_agent_tv,
    n_hands=1000,
    seed_start=80000
)

fixed_vs_tv_df

In [ ]:
x = np.arange(len(fixed_vs_tv_df["Opponent Pool"]))
width=0.35

plt.figure(figsize=(12, 5))
plt.bar(
    x-width/2,
    fixed_vs_tv_df["Fixed Agent Avg Reward"],
    width,
    label="Fixed-pool trained Q agent"
)
plt.bar(
    x+width/2,
    fixed_vs_tv_df["TV Agent Avg Reward"],
    width,
    label="Time-varying trained Q agent"
)

plt.xticks(x, fixed_vs_tv_df["Opponent Pool"], rotation=30)
plt.xlabel("Opponent Pool")
plt.ylabel("Average Reward")
plt.title("Fixed-pool Q Agent vs Time-varying Q Agent")
plt.legend()
plt.show()

## 7. Adaptive Q-learning Agent
The adaptive version below keeps the trained base Q-table fixed and adds a behavior-tracking layer. The aim is to adjust action preferences when the recent opponent pool appears passive, aggressive, bluff-heavy, tight, random, or mixed.


### 7.1 Opponent behavior tracker
The tracker records recent fold, check/call, and bet/raise behavior, then classifies individual players and the overall opponents pool.


In [ ]:
class OpponentBehaviorTracker:

    def __init__(self, n_players, window_size=100, min_actions=10):
        self.n_players = n_players
        self.window_size = window_size
        self.min_actions = min_actions

        self.actions = {
            player_id: deque(maxlen=window_size)
            for player_id in range(n_players)
        }

        self.showdown_results = {
            player_id: deque(maxlen=window_size)
            for player_id in range(n_players)
        }

        # keep compatibility with the previous version.  (rolling-window counts, not full-history counts
        self.showdown_count = {
            player_id: 0
            for player_id in range(n_players)
        }

        self.showdown_win_count = {
            player_id: 0
            for player_id in range(n_players)
        }

    def update_from_action(self, player_id, action_name):
        ignored_actions = {"small_blind", "big_blind"}

        if action_name in ignored_actions:
            return

        self.actions[player_id].append(action_name)

    def update_from_env_last_action(self, env):
        if len(env.action_history) == 0:
            return

        last_action = env.action_history[-1]

        player_id = last_action["player"]
        action_name = last_action["action"]

        self.update_from_action(player_id, action_name)

    def update_after_hand(self, env):
        if env.terminal_reason != "showdown":
            return

        active_players = [
            p.player_id for p in env.players
            if not p.folded
        ]

        for player_id in active_players:
            result = int(player_id in env.winner_ids)

            self.showdown_results[player_id].append(result)

            self.showdown_count[player_id] = len(
                self.showdown_results[player_id]
            )

            self.showdown_win_count[player_id] = sum(
                self.showdown_results[player_id]
            )

    def get_player_stats(self, player_id):
        action_list = list(self.actions[player_id])
        n_actions = len(action_list)

        if n_actions == 0:
            return {
                "n_actions": 0,
                "fold_rate": 0,
                "check_call_rate": 0,
                "bet_raise_rate": 0,
                "showdown_win_rate": None
            }

        fold_count = sum(a == "fold" for a in action_list)

        check_call_count = sum(
            a in {"check", "call", "all_in_call"}
            for a in action_list
        )

        bet_raise_count = sum(
            a in {"bet", "raise"}
            for a in action_list
        )

        showdown_list = list(self.showdown_results[player_id])

        if len(showdown_list)>0:
            showdown_win_rate = np.mean(showdown_list)

        else:
            showdown_win_rate = None

        return {
            "n_actions": n_actions,
            "fold_rate": fold_count / n_actions,
            "check_call_rate": check_call_count / n_actions,
            "bet_raise_rate": bet_raise_count / n_actions,
            "showdown_win_rate": showdown_win_rate
        }

    def classify_player(self, player_id):
        """
        Classify one player's style based on recent behavior.
        """
        stats = self.get_player_stats(player_id)

        if stats["n_actions"] < self.min_actions:
            return "unknown"

        fold_rate = stats["fold_rate"]
        check_call_rate = stats["check_call_rate"]
        bet_raise_rate = stats["bet_raise_rate"]
        showdown_win_rate = stats["showdown_win_rate"]

        # high aggression but poor showdown performance can indicate bluff-heavy behavior.
        if bet_raise_rate >= 0.35:
            if showdown_win_rate is not None and showdown_win_rate < 0.4:
                return "bluff_heavy"

            return "aggressive"

        # frequent checking/calling with little raising indicates passive style.
        if check_call_rate >= 0.65 and bet_raise_rate < 0.2:
            return "passive"

        # frequent folding with little raising indicates tight style.
        if fold_rate >= 0.45 and bet_raise_rate < 0.2:
            return "tight"

        # if no clear pattern appears, treat it as random/mixed.
        return "random"

    def get_pool_stats(self, player_ids):
        all_actions = []
        all_showdown_results = []

        for player_id in player_ids:
            all_actions.extend(list(self.actions[player_id]))
            all_showdown_results.extend(list(self.showdown_results[player_id]))

        n_actions = len(all_actions)

        if n_actions==0:
            return {
                "n_actions": 0,
                "fold_rate": 0,
                "check_call_rate": 0,
                "bet_raise_rate": 0,
                "showdown_win_rate": None
            }

        fold_count = sum(a == "fold" for a in all_actions)

        check_call_count = sum(
            a in {"check", "call", "all_in_call"}
            for a in all_actions
        )

        bet_raise_count = sum(
            a in {"bet", "raise"}
            for a in all_actions
        )

        if len(all_showdown_results)>0:
            showdown_win_rate = np.mean(all_showdown_results)

        else:
            showdown_win_rate = None

        return {
            "n_actions": n_actions,
            "fold_rate": fold_count / n_actions,
            "check_call_rate": check_call_count / n_actions,
            "bet_raise_rate": bet_raise_count / n_actions,
            "showdown_win_rate": showdown_win_rate
        }

    def classify_pool(self, player_ids):
        stats = self.get_pool_stats(player_ids)

        if stats["n_actions"] < self.min_actions:
            return "unknown"

        fold_rate = stats["fold_rate"]
        check_call_rate = stats["check_call_rate"]
        bet_raise_rate = stats["bet_raise_rate"]
        showdown_win_rate = stats["showdown_win_rate"]

        # bluff-heavy: relatively high aggression, but not necessarily strong showdown result.
        if bet_raise_rate >= 0.22:
            if showdown_win_rate is not None and showdown_win_rate < 0.45:
                return "bluff_heavy"

            if fold_rate >= 0.25:
                return "bluff_heavy"

            return "aggressive"

        # aggressive but slightly below the main threshold.
        if bet_raise_rate >= 0.18 and check_call_rate < 0.7:
            return "aggressive"

        # passive: many checks/calls and very little betting/raising.
        if check_call_rate >= 0.65 and bet_raise_rate < 0.18:
            return "passive"

        # tight: folds a lot and rarely raises.
        if fold_rate >= 0.4 and bet_raise_rate < 0.18:
            return "tight"

        return "mixed"

    def print_summary(self):
        print("=" *80)
        print("Opponent Behavior Tracker Summary")
        print("=" *80)

        for player_id in range(self.n_players):
            stats = self.get_player_stats(player_id)
            label = self.classify_player(player_id)

            print(
                f"Player {player_id} | "
                f"style={label:12s} | "
                f"n={stats['n_actions']:4d} | "
                f"fold={stats['fold_rate']:.3f} | "
                f"check/call={stats['check_call_rate']:.3f} | "
                f"bet/raise={stats['bet_raise_rate']:.3f} | "
                f"showdown_win_rate={stats['showdown_win_rate']}"
            )

        print("=" *80)

### 7.2 Adaptive policy layer
The adaptive agent starts from the learned Q-values and applies small style-dependent adjustments before selecting the best legal action.


In [ ]:
class AdaptiveQLearningAgent(BaseAgent):

    def __init__(
        self,
        base_q_agent,
        config,
        name="AdaptiveQLearningAgent",
        adjustment_strength=0.4
    ):
        super().__init__(name=name)

        self.base_q_agent = base_q_agent
        self.config = config
        self.adjustment_strength = adjustment_strength

    def select_action(self, obs, tracker, opponent_player_ids):
        state = encode_state(obs, self.config)
        legal_actions = list(obs["legal_actions"])

        if len(legal_actions)==0:
            raise ValueError("No legal actions available.")

        q_values = self.base_q_agent.get_q_values(state).copy()

        opponent_style = tracker.classify_pool(opponent_player_ids)

        hand_strength = estimate_simple_hand_strength(
            obs["hole_cards"],
            obs["community_cards"]
        )

        to_call = obs["to_call"]
        raise_available = 2 in legal_actions
        fold_available = 0 in legal_actions
        call_available = 1 in legal_actions

        q_values = self._adjust_q_values(
            q_values=q_values,
            opponent_style=opponent_style,
            hand_strength=hand_strength,
            to_call=to_call,
            raise_available=raise_available,
            fold_available=fold_available,
            call_available=call_available
        )

        legal_q_values = {
            action: q_values[action]
            for action in legal_actions
        }

        max_q = max(legal_q_values.values())

        best_actions = [
            action for action, q in legal_q_values.items()
            if q == max_q
        ]

        action = random.choice(best_actions)

        debug_info = {
            "state": state,
            "opponent_style": opponent_style,
            "hand_strength": hand_strength,
            "adjusted_q_values": q_values,
            "selected_action": action
        }

        return action, debug_info

    def _adjust_q_values(
        self,
        q_values,
        opponent_style,
        hand_strength,
        to_call,
        raise_available,
        fold_available,
        call_available
    ):
        adj = self.adjustment_strength

        # Against passive opponents:
        # passive players call too much and rarely punish aggression.
        # Therefore, value betting stronger hands becomes more attractive.
        if opponent_style == "passive":
            if raise_available and hand_strength >= 2:
                q_values[2] += adj

            if raise_available and to_call == 0 and hand_strength >= 1:
                q_values[2] += 0.5*adj

            if fold_available:
                q_values[0] -= 0.3*adj

        # Against aggressive opponents:
        # Avoid over-bluffing with weak hands.
        # Fold more weak hands facing bets, but continue with medium/strong hands.
        elif opponent_style == "aggressive":
            if to_call > 0:
                if hand_strength == 0 and fold_available:
                    q_values[0] += adj

                if hand_strength in [1, 2] and call_available:
                    q_values[1] += 0.6*adj

                if hand_strength >= 3 and call_available:
                    q_values[1] += 0.3*adj

            if raise_available and hand_strength <= 1:
                q_values[2] -= adj

        # Against bluff-heavy opponents:
        # call more often with non-trash hands because opponents may be over-bluffing
        elif opponent_style == "bluff_heavy":
            if to_call > 0:
                if hand_strength >= 1 and call_available:
                    q_values[1] += adj

                if hand_strength == 0 and fold_available:
                    q_values[0] += 0.5*adj

            if raise_available and hand_strength >= 3:
                q_values[2] += 0.5 \*adj

        # Against tight opponents:
        # respect their aggression more, but steal more when no bet is faced
        elif opponent_style == "tight":
            if to_call > 0 and hand_strength <= 1 and fold_available:
                q_values[0] += adj

            if to_call == 0 and raise_available and hand_strength >= 1:
                q_values[2] += 0.5*adj

        # Against random or mixed opponents:
        # use mild value-oriented adjustment
        elif opponent_style in {"random", "mixed"}:
            if raise_available and hand_strength >= 3:
                q_values[2] += 0.4*adj

            if to_call > 0 and hand_strength == 0 and fold_available:
                q_values[0] += 0.3*adj

        # For unknown style:
        # not adjust much, rely mostly on the base Q policy.
        elif opponent_style == "unknown":
            pass

        return q_values

### 7.3 Adaptive evaluation under time-varying opponents

The adaptive agent is evaluated for 20,000 hands under the same time-varying phase schedule. This can allow a direct comparison with fixed Q-learning policy in the later cells.


In [ ]:
def evaluate_adaptive_agent_time_varying_pool(
    base_q_agent,
    n_hands=20000,
    seed_start=90000,
    tracker_window_size=100,
    adjustment_strength=0.4,
    verbose_every=500
):
    config = PokerConfig(
        n_players=4,
        starting_stack=100,
        small_blind=1,
        big_blind=2,
        fixed_bet=2,
        max_raises_per_round=2
    )

    env = FixedLimitTexasHoldemEnv(config)

    adaptive_agent = AdaptiveQLearningAgent(
        base_q_agent=base_q_agent,
        config=config,
        adjustment_strength=adjustment_strength
    )

    tracker = OpponentBehaviorTracker(
        n_players=config.n_players,
        window_size=tracker_window_size,
        min_actions=10
    )

    rl_player_id =0
    opponent_player_ids = [1,2,3]

    old_epsilon = base_q_agent.epsilon
    base_q_agent.set_epsilon(0.0)

    rewards = []
    wins = []
    phases = []
    detected_styles = []
    terminal_reasons = []

    for hand_idx in range(n_hands):
        phase = get_phase_by_episode(hand_idx)
        opponent_agents = create_opponent_pool(phase)

        random.seed(seed_start + hand_idx)
        env.reset(seed=seed_start + hand_idx, rotate_dealer=True)

        last_detected_style = "unknown"

        while not env.done:
            current_player = env.current_player
            obs = env.get_observation(current_player)

            if current_player == rl_player_id:
                action, debug_info = adaptive_agent.select_action(
                    obs=obs,
                    tracker=tracker,
                    opponent_player_ids=opponent_player_ids
                )
                last_detected_style = debug_info["opponent_style"]

            else:
                opponent = opponent_agents[current_player]
                action = opponent.act(obs)

            env.step(action)

            # update tracker using the latest observed action.
            tracker.update_from_env_last_action(env)

        tracker.update_after_hand(env)

        rewards.append(env.last_rewards[rl_player_id])
        wins.append(int(rl_player_id in env.winner_ids))
        phases.append(phase)
        detected_styles.append(last_detected_style)
        terminal_reasons.append(env.terminal_reason)

        if verbose_every is not None and (hand_idx + 1) % verbose_every == 0:
            recent_reward = np.mean(rewards[-verbose_every:])
            recent_win_rate = np.mean(wins[-verbose_every:])

            print(
                f"Hand {hand_idx + 1}/{n_hands} | "
                f"True phase: {phase} | "
                f"Detected style: {last_detected_style} | "
                f"Recent avg reward: {recent_reward:.3f} | "
                f"Recent win rate: {recent_win_rate:.3f}"
            )

    base_q_agent.set_epsilon(old_epsilon)

    history = {
        "rewards": rewards,
        "wins": wins,
        "phases": phases,
        "detected_styles": detected_styles,
        "terminal_reasons": terminal_reasons,
        "tracker": tracker
    }

    return adaptive_agent, history

In [ ]:
adaptive_agent, adaptive_history = evaluate_adaptive_agent_time_varying_pool(
    base_q_agent=q_agent,
    n_hands=20000,
    seed_start=90000,
    tracker_window_size=100,
    adjustment_strength=0.4,
    verbose_every=500
)

def summarize_adaptive_history(history):
    df = pd.DataFrame({
        "reward": history["rewards"],
        "win": history["wins"],
        "phase": history["phases"],
        "detected_style": history["detected_styles"],
        "terminal_reason": history["terminal_reasons"]
    })

    phase_summary = (
        df.groupby("phase")
        .agg(
            avg_reward=("reward", "mean"),
            reward_std=("reward", "std"),
            win_rate=("win", "mean"),
            n_hands=("reward", "count")
        )
        .reset_index()
    )

    style_summary = (
        df.groupby(["phase", "detected_style"])
        .size()
        .reset_index(name="count")
        .sort_values(["phase", "count"], ascending=[True,False])
    )

    return phase_summary, style_summary, df

adaptive_phase_summary, adaptive_style_summary, adaptive_detail_df = summarize_adaptive_history(
    adaptive_history
)

adaptive_phase_summary

adaptive_style_summary

In [ ]:
adaptive_rewards = adaptive_history["rewards"]

plt.figure(figsize=(12,5))
plt.plot(moving_average(adaptive_rewards, window=1000), label="1000-hand moving average")

plt.axvline(5000, linestyle="--", label="Phase change")
plt.axvline(10000, linestyle="--")
plt.axvline(15000, linestyle="--")

plt.xlabel("Hand")
plt.ylabel("Reward")
plt.title("Adaptive Q Agent Reward under Time-varying Opponent Pool")
plt.legend()
plt.show()

In [ ]:
adaptive_wins = adaptive_history["wins"]

plt.figure(figsize=(12, 5))
plt.plot(moving_average(adaptive_wins, window=1000), label="1000-hand moving win rate")

plt.axvline(5000, linestyle="--", label="Phase change")
plt.axvline(10000, linestyle="--")
plt.axvline(15000, linestyle="--")

plt.xlabel("Hand")
plt.ylabel("Win Rate")
plt.title("Adaptive Q Agent Win Rate under Time-varying Opponent Pool")
plt.legend()
plt.show()

In [ ]:
#evaluate a fixed Q-learning agent under the same time-varying opponent pool without behavior tracking or adaptive adjustment
def evaluate_fixed_q_agent_time_varying_pool(
    q_agent,
    n_hands=4000,
    seed_start=90000,
    verbose_every=500
):
    config = PokerConfig(
        n_players=4,
        starting_stack=100,
        small_blind=1,
        big_blind=2,
        fixed_bet=2,
        max_raises_per_round=2
    )

    env = FixedLimitTexasHoldemEnv(config)

    rl_player_id = 0

    old_epsilon = q_agent.epsilon
    q_agent.set_epsilon(0.0)

    rewards = []
    wins = []
    phases = []
    terminal_reasons = []

    for hand_idx in range(n_hands):
        phase = get_phase_by_episode(hand_idx)
        opponent_agents = create_opponent_pool(phase)

        random.seed(seed_start + hand_idx)
        env.reset(seed=seed_start + hand_idx, rotate_dealer=True)

        while not env.done:
            current_player = env.current_player
            obs = env.get_observation(current_player)

            if current_player == rl_player_id:
                state = encode_state(obs, env.config)

                action = q_agent.select_action(
                    state=state,
                    legal_actions=obs["legal_actions"],
                    training=False
                )

            else:
                opponent = opponent_agents[current_player]
                action = opponent.act(obs)

            env.step(action)

        rewards.append(env.last_rewards[rl_player_id])
        wins.append(int(rl_player_id in env.winner_ids))
        phases.append(phase)
        terminal_reasons.append(env.terminal_reason)

        if verbose_every is not None and (hand_idx + 1) % verbose_every == 0:
            recent_reward = np.mean(rewards[-verbose_every:])
            recent_win_rate = np.mean(wins[-verbose_every:])

            print(
                f"Hand {hand_idx + 1}/{n_hands} | "
                f"Phase: {phase} | "
                f"Recent avg reward: {recent_reward:.3f} | "
                f"Recent win rate: {recent_win_rate:.3f}"
            )

    q_agent.set_epsilon(old_epsilon)

    history = {
        "rewards": rewards,
        "wins": wins,
        "phases": phases,
        "terminal_reasons": terminal_reasons
    }

    return history

fixed_tv_eval_history = evaluate_fixed_q_agent_time_varying_pool(
    q_agent=q_agent,
    n_hands=20000,
    seed_start=90000,
    verbose_every=500
)

def compare_fixed_and_adaptive_histories(fixed_history, adaptive_history):
    fixed_df = pd.DataFrame({
        "reward": fixed_history["rewards"],
        "win": fixed_history["wins"],
        "phase": fixed_history["phases"],
        "agent": "Fixed Q Agent"
    })

    adaptive_df = pd.DataFrame({
        "reward": adaptive_history["rewards"],
        "win": adaptive_history["wins"],
        "phase": adaptive_history["phases"],
        "agent": "Adaptive Q Agent"
    })

    df = pd.concat([fixed_df, adaptive_df], ignore_index=True)

    summary = (
        df.groupby(["agent", "phase"])
        .agg(
            avg_reward=("reward", "mean"),
            reward_std=("reward", "std"),
            win_rate=("win", "mean"),
            n_hands=("reward", "count")
        )
        .reset_index()
    )

    return summary, df

fixed_vs_adaptive_summary, fixed_vs_adaptive_detail_df = compare_fixed_and_adaptive_histories(
    fixed_history=fixed_tv_eval_history,
    adaptive_history=adaptive_history
)

fixed_vs_adaptive_summary


In [ ]:
pivot_reward = fixed_vs_adaptive_summary.pivot(
    index="phase",
    columns="agent",
    values="avg_reward"
)

pivot_reward.plot(kind="bar", figsize=(10,5))

plt.xlabel("Opponent Pool Phase")
plt.ylabel("Average Reward")
plt.title("Fixed Q Agent vs Adaptive Q Agent: Average Reward")
plt.xticks(rotation=30)
plt.show()

In [ ]:
pivot_win = fixed_vs_adaptive_summary.pivot(
    index="phase",
    columns="agent",
    values="win_rate"
)

pivot_win.plot(kind="bar", figsize=(10,5))

plt.xlabel("Opponent Pool Phase")
plt.ylabel("Win Rate")
plt.title("Fixed Q Agent vs Adaptive Q Agent: Win Rate")
plt.xticks(rotation=30)
plt.show()